# Health_plan_craw

In [8]:
!pip install selenium webdriver-manager beautifulsoup4 pandas

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.5 MB 3.4 MB/s eta 0:00:03
   ----- ---------------------------------- 1.3/9.5 MB 3.2 MB/s eta 0:00:03
   ----- ---------------------------------- 1.3/9.5 MB 3.2 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/9.5 MB 2.2 MB/s eta 0:00:04
   ---------- ----------------------------- 2.6/9.5 MB 2.3 MB/s eta 0:00:04
   ---------- ----------------------------- 2.6/9.5 MB 2.3 MB/s eta 0:00:04
   ------------- -------------------------- 3.1/9.5 MB 2.1 MB/s eta 0:00:03
   --------------- ------------------------ 3.7/9.5 MB 2.2 MB/s eta 0:00:03
   ----------------- ---------------------- 4.2/9.5 MB 2.1 MB/s eta 0:00:03
   --------------------- ------------------ 5.2/9.5 MB 2.5 MB/s eta 0:00:02
   --------------------------- ------------ 6.6/9.5 MB 2.7 MB/s eta 0:00:02
   -----------------------

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time

url = "https://tricare.mil/Costs/Compare"


def crawl_all_tabs():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")           # 브라우저 창 숨기기
    options.add_argument("--no-sandbox")            # 리눅스 환경에서 권한 문제로 크롤러가 죽는 걸 방지
    options.add_argument("--disable-dev-shm-usage") # 메모리가 부족한 환경에서 크래시 방지
    options.add_argument("--window-size=1920,1080")

    # ChromeDriver 자동 설치 및 실행
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    driver.get(url)     # 브라우저로 해당 url 열림
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "button[role='tab']"))
    )

    all_data = []

    # 탭 버튼 목록 가져오기
    tabs = driver.find_elements(By.CSS_SELECTOR, "button[role='tab']")
    num_tabs = len(tabs)
    print(f"탭 개수: {num_tabs}개")

    for i in range(num_tabs):
        # 탭 목록 다시 가져오기 (DOM 갱신 대응)
        tabs = driver.find_elements(By.CSS_SELECTOR, "button[role='tab']")
        tab = tabs[i]
        tab_name = tab.text.strip().replace("\n", " ")
        print(f"[{i+1}/{num_tabs}] 처리 중: {tab_name}")

        # 탭 클릭
        driver.execute_script("arguments[0].click();", tab)
        time.sleep(1)  # 표 렌더링 대기

        # 현재 활성 패널 찾기
        panel_id = tab.get_attribute("aria-controls")
        panel = driver.find_element(By.ID, panel_id)

        # 컬럼 헤더 추출
        soup = BeautifulSoup(panel.get_attribute("innerHTML"), "html.parser")
        headers = [th.get_text(" ", strip=True) for th in soup.select("table thead tr th")]

        # 표 행 파싱
        rows = soup.select("table tbody tr")
        for row in rows:
            cells = row.find_all("td")
            if not cells:
                continue

            row_data = {"탭": tab_name}

            # 헤더가 있으면 헤더명으로, 없으면 기본 컬럼명으로
            col_names = headers if headers else ["비용 유형", "플랜 1", "플랜 2"]
            for j, cell in enumerate(cells):
                col = col_names[j] if j < len(col_names) else f"컬럼{j+1}"
                row_data[col] = cell.get_text(" ", strip=True)

            all_data.append(row_data)

    driver.quit()
    return all_data


if __name__ == "__main__":
    print("크롤링 시작...")
    data = crawl_all_tabs()

    if not data:
        print("수집된 데이터가 없습니다. 셀렉터 또는 페이지 구조를 확인하세요.")
    else:
        df = pd.DataFrame(data)
        output_file = "Health_Plan_Costs.csv"
        df.to_csv(output_file, index=False, encoding="utf-8-sig")
        print(f"\n저장 완료: {len(df)}행 → {output_file}")
        print(df.head())

크롤링 시작...
탭 개수: 8개
[1/8] 처리 중: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
[2/8] 처리 중: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group B
[3/8] 처리 중: TRICARE Prime & TRICARE Select Retirees, their family members & all others Group A
[4/8] 처리 중: TRICARE Prime & TRICARE Select Retirees, their family members & all others Group B
[5/8] 처리 중: TRICARE For Life
[6/8] 처리 중: TRICARE Reserve Select
[7/8] 처리 중: TRICARE Retired Reserve
[8/8] 처리 중: TRICARE Young Adult

저장 완료: 234행 → Health_Plan_Costs.csv
                                                   탭  \
0  TRICARE Prime & TRICARE Select Active duty fam...   
1  TRICARE Prime & TRICARE Select Active duty fam...   
2  TRICARE Prime & TRICARE Select Active duty fam...   
3  TRICARE Prime & TRICARE Select Active duty fam...   
4  TRICARE Prime & TRICARE Select Active duty fam...   

                             비용 유형  \
0          Enrollment fee (annual)   
1  

In [11]:
df=pd.read_csv('Health_Plan_Costs.csv')
df

,탭,비용 유형,플랜 1,플랜 2,컬럼4
0,TRICARE Prime & TRICARE Select Active duty fam...,Enrollment fee (annual),$0,$0,NaN
1,TRICARE Prime & TRICARE Select Active duty fam...,Deductible (annual),$0,E-1 through E-4: $50 per individual $100 per f...,NaN
2,TRICARE Prime & TRICARE Select Active duty fam...,Catastrophic cap (annual),"$1,000 per family","$1,000 per family",NaN
3,TRICARE Prime & TRICARE Select Active duty fam...,Clinical preventive services,Network: $0 Non-network: Point-of-service fees,Network and Non-network: $0,NaN
4,TRICARE Prime & TRICARE Select Active duty fam...,Outpatient visit — primary care,Network: $0 Non-network: Point-of-service fees,Network: $28 Non-network: 20%,NaN
...,...,...,...,...,...
229,TRICARE Young Adult,Maternity — hospital delivery,Network: $198 per admission,Network: $231 per admission Non-network: 25%,NaN
230,TRICARE Young Adult,Maternity — birthing center delivery,Network: $79 Non-network: Point-of-service fees,Network: $125 per admission Non-network: 25%,NaN
231,TRICARE Young Adult,Maternity — delivery at home (primary care),Network: $26 Non-network: Point-of-service fees,Network: $33 Non-network: 25%,NaN
232,TRICARE Young Adult,Maternity — delivery at home (specialty care),Network: $39 Non-network: Point-of-service fees,Network: $52 Non-network: 25%,NaN
